# Lab 02: Model Providers

**Goal:** swap models without changing agent code, tune model parameters, and get structured output.

Course module: `samples/02-model-providers`

Strands is model-agnostic. The `Agent` takes any `Model`, so the loop, tools, and hooks stay the same when you switch providers.

In [ ]:
# Run this first in every lab: puts the repo root on sys.path and loads lab_config
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from lab_config import get_model, describe
print(describe())

## 1. Constructing each provider explicitly
`lab_config.get_model()` wraps these. Here they are directly (from `model_providers.py`). Each cell only runs if that provider is available to you.

In [ ]:
import os
from strands import Agent

available = {}

# --- Amazon Bedrock (uses your AWS credentials / boto3 session)
try:
    import boto3
    boto3.client("sts").get_caller_identity()
    from strands.models import BedrockModel
    available["bedrock"] = BedrockModel(
        # model_id="us.anthropic.claude-sonnet-5",   # pick any Bedrock model or inference profile
        region_name=os.environ.get("AWS_REGION", "us-east-1"),
        temperature=0.3,
    )
except Exception as e:
    print("bedrock unavailable:", type(e).__name__)

# --- Anthropic API
if os.environ.get("ANTHROPIC_API_KEY"):
    from strands.models.anthropic import AnthropicModel
    available["anthropic"] = AnthropicModel(
        client_args={"api_key": os.environ["ANTHROPIC_API_KEY"]},
        model_id="claude-sonnet-5-5",
        max_tokens=1024,
        params={"temperature": 0.3},
    )

# --- OpenAI
if os.environ.get("OPENAI_API_KEY"):
    from strands.models.openai import OpenAIModel
    available["openai"] = OpenAIModel(
        client_args={"api_key": os.environ["OPENAI_API_KEY"]},
        model_id="gpt-4o",
        params={"max_tokens": 1000, "temperature": 0.3},
    )

# --- Ollama (local)
try:
    import urllib.request; urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
    from strands.models.ollama import OllamaModel
    available["ollama"] = OllamaModel(
        host="http://localhost:11434",
        model_id=(os.environ.get("LAB_PROVIDER") == "ollama" and os.environ.get("LAB_MODEL_ID")) or "gemma4:latest",
        temperature=0.3,
    )
except Exception as e:
    print("ollama unavailable:", type(e).__name__)

print("Available providers:", list(available))

## 2. Same agent code, different models

In [ ]:
import time

PROMPT = "Explain the agent loop in exactly two sentences."

for name, model in available.items():
    agent = Agent(model=model, callback_handler=None)
    t0 = time.time()
    try:
        r = agent(PROMPT)
    except Exception as e:
        print(f"--- {name}: failed ({type(e).__name__}: {e})\n"); continue
    usage = r.metrics.get_summary()["accumulated_usage"]
    print(f"--- {name}  ({time.time()-t0:.1f}s, {usage.get('totalTokens')} tokens)")
    print(str(r).strip(), "\n")

## 3. Changing model config at runtime
`model.update_config(...)` changes parameters on an existing model, for example to lower temperature for deterministic tasks. `model.get_config()` shows the current values.

In [ ]:
model = get_model()
print("before:", model.get_config())
model.update_config(temperature=0.0)
print("after: ", model.get_config())

## 4. Structured output
Pass a Pydantic model and Strands forces the response into that schema, validated and typed. This works with any provider.

In [ ]:
from pydantic import BaseModel, Field

class ProviderFacts(BaseModel):
    name: str = Field(description="Provider name")
    hosted: bool = Field(description="True if it's a hosted cloud API, False if it runs locally")
    one_line_pitch: str

agent = Agent(model=get_model(), callback_handler=None)
result = agent(
    "Give facts about Ollama as an LLM provider.",
    structured_output_model=ProviderFacts,
)
facts: ProviderFacts = result.structured_output
print(repr(facts))

## 🧪 Exercise
- If you have two providers, run the same tool-using agent from Lab 01 on both. Compare `total_cycles` and token counts.
- Throttled on Bedrock? Try `BedrockModel(model_id="us.amazon.nova-lite-v1:0")`.

**Next:** [Lab 03: Tools & MCP](03-tools-and-mcp.ipynb)

In [ ]:
# your code here